# BirdCLEF+ 2026: Multi-Model Acoustic Species Identification
## Pantanal Wetlands Passive Acoustic Monitoring Pipeline

---

**Competition**: [BirdCLEF+ 2026](https://kaggle.com/competitions/birdclef-2026)  
**Objective**: Identify 234 wildlife species (birds, amphibians, mammals, reptiles, insects) from continuous audio recordings across the Pantanal wetlands.  
**Metric**: Macro-averaged ROC-AUC (skipping classes with no positives in test).  
**Constraint**: CPU-only notebook, 90-minute runtime, no internet access.


## Table of Contents

1. [Environment Setup](#setup)
2. [Configuration](#config)
3. [Data Loading and Exploratory Analysis](#eda)
4. [Label Preprocessing and Truth Matrix](#labels)
5. [Perch v2 Model and Species Mapping](#perch)
6. [Perch Inference Engine](#engine)
7. [Bayesian Prior Fusion](#priors)
8. [Out-of-Fold Meta-Features](#oof)
9. [Embedding Probes](#probes)
10. [Test Inference and Submission](#submit)


---

## Approach Overview

The pipeline leverages Google Perch v2 as a frozen feature extractor. No fine-tuning is performed on the pretrained model. Instead, the 59 fully-labeled soundscape files provide sufficient signal to train lightweight classifiers on frozen embeddings.

```
Perch v2 (frozen, 14k-class pretrained)
    |
    +-- raw logits (234 mapped classes)
    |       |
    |       +-- site x hour Bayesian prior fusion
    |                |
    |                v
    |           fused logits (base scores)
    |
    +-- 1536-dim embeddings
            |
            v
        PCA (64d) + per-class MLP probes
            |
            v
        final = (1-alpha) * base + alpha * probe -> submission.csv
```

The species taxonomy split between event-type sounds (Aves) and texture-type sounds (Amphibia, Insecta) drives different fusion weights and temporal smoothing strategies. Frogs and insects produce continuous choruses whose presence correlates strongly with site and time-of-day. Bird calls are transient events where the Perch logit carries more weight than the spatial prior.


---

# 1. Environment Setup <a name="setup"></a>

---

TensorFlow 2.20 is required because the Perch v2 SavedModel uses StableHLO operations introduced in that release. The Kaggle default is 2.19, so pre-staged wheels are installed from an attached dataset. This avoids any internet dependency during submission.


In [ ]:
import subprocess, sys
from pathlib import Path

# Perch v2 SavedModel uses StableHLO ops introduced in TF 2.20
# Kaggle default is 2.19, so install from pre-staged wheels
_WHL = Path("/kaggle/input/tensorflow-2-20-wheels")
if not _WHL.exists():
    _WHL = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel")

if _WHL.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
         str(_WHL / "tensorboard-2.20.0-py3-none-any.whl")],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
         str(_WHL / "tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl")],
        check=True,
    )
    print("Installed TF 2.20.0 from local wheels.")
else:
    print("Wheel directory not found. Using system TensorFlow.")


## Libraries


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # force CPU per competition rules

import gc
import json
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()

# Visual defaults for consistent plot styling
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi": 100,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")


---

# 2. Configuration <a name="config"></a>

---

All paths and hyperparameters are centralized in one `Settings` class. The `MODE` flag controls whether expensive OOF grid searches run (train) or frozen parameters are used (submit). Cache auto-detection searches candidate directories and loads the first valid hit, which keeps the notebook compatible across local dev and Kaggle environments.


In [ ]:
class Settings:
    # 'submit' uses frozen hyperparameters
    # 'train'  runs probe grid search and prints OOF diagnostics
    MODE = "submit"
    SEED = 42

    # Competition paths
    _KAGGLE = Path("/kaggle/input/competitions/birdclef-2026")
    _LOCAL  = Path("../dataset")
    BASE    = _KAGGLE if _KAGGLE.exists() else _LOCAL

    # Perch v2 SavedModel
    MODEL_DIR = Path(
        "/kaggle/input/models/google/bird-vocalization-classifier"
        "/tensorflow2/perch_v2_cpu/1"
    )

    # Perch cache (pre-computed embeddings to save submission runtime)
    _CACHE_DIRS = [
        Path("/kaggle/input/perch-meta"),
        Path("/kaggle/input/datasets/jaejohn/perch-meta"),
        Path("/kaggle/working/cache"),
    ]
    CACHE_DIR = next(
        (d for d in _CACHE_DIRS
         if (d / "full_perch_meta.parquet").exists()
         and (d / "full_perch_arrays.npz").exists()),
        None,
    )
    WORK_CACHE = Path("/kaggle/working/cache")

    # Audio constants -- all recordings are standardized to 32kHz
    SR             = 32_000
    WINDOW_SEC     = 5
    WINDOW_SAMPLES = SR * WINDOW_SEC   # 160,000 samples per window
    FILE_SAMPLES   = 60 * SR           # 1,920,000 samples per file
    N_WINDOWS      = 12                # twelve 5-second windows per minute
    BATCH_FILES    = 16
    DRYRUN_FILES   = 20

    # Prior fusion lambdas (tuned via OOF on labeled soundscapes)
    # Birds are transient events with reliable Perch logits
    LAMBDA_EVENT         = 0.4
    # Frogs and insects are location-determined recurring textures
    LAMBDA_TEXTURE       = 1.0
    # Genus proxy is noisier than a direct species mapping
    LAMBDA_PROXY_TEXTURE = 0.8
    SMOOTH_TEXTURE       = 0.35
    SMOOTH_EVENT         = 0.15

    # Embedding probe hyperparameters (frozen from OOF grid search)
    PCA_DIM     = 64
    MIN_POS     = 8
    PROBE_C     = 0.50
    PROBE_ALPHA = 0.40

    # MLP probe parameters
    MLP_PARAMS = dict(
        hidden_layer_sizes=(128,),
        activation="relu",
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=42,
        learning_rate_init=0.001,
        alpha=0.01,
    )


CFG = Settings()
CFG.WORK_CACHE.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=42):
    """Pin all random seeds for full reproducibility."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(CFG.SEED)

print(f"MODE       : {CFG.MODE}")
print(f"BASE       : {CFG.BASE}")
print(f"CACHE      : {CFG.CACHE_DIR}")


---

# 3. Data Loading and Exploratory Analysis <a name="eda"></a>

---

Four metadata files are loaded: taxonomy (234 species), train recordings (~35k), soundscape annotations, and the sample submission. The soundscape annotations contain duplicate rows which are de-duplicated before building the truth matrix.


In [ ]:
taxonomy        = pd.read_csv(CFG.BASE / "taxonomy.csv")
train_meta      = pd.read_csv(CFG.BASE / "train.csv")
soundscape_raw  = pd.read_csv(CFG.BASE / "train_soundscapes_labels.csv")
sample_sub      = pd.read_csv(CFG.BASE / "sample_submission.csv")

# Drop exact duplicate rows in soundscape annotations
soundscape_lbls = soundscape_raw.drop_duplicates().reset_index(drop=True)

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

print(f"Taxonomy species    : {len(taxonomy)}")
print(f"Train recordings    : {len(train_meta):,}")
print(f"Soundscape rows     : {len(soundscape_lbls):,} unique "
      f"(dropped {len(soundscape_raw) - len(soundscape_lbls):,} duplicates)")
print(f"Submission columns  : {N_CLASSES}")


## Taxonomic Class Distribution

The 234 target species span five animal classes. Understanding this breakdown is essential because the acoustic properties of each class inform our fusion and smoothing strategies. Bird vocalizations are discrete events, while frog choruses and insect stridulation produce continuous textures.


In [ ]:
class_counts = taxonomy["class_name"].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12", "#9b59b6"]
class_counts.plot.barh(ax=ax, color=colors[:len(class_counts)])
ax.set_xlabel("Number of Species")
ax.set_title("Species Count by Taxonomic Class")
for i, v in enumerate(class_counts.values):
    ax.text(v + 0.5, i, str(v), va="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


## Training Recording Distribution

The training data consists of short focal recordings from Xeno-canto and iNaturalist. Coverage varies dramatically across species, which directly impacts probe training. Species with fewer than 8 positive windows in the labeled soundscapes cannot support reliable per-class probes.


In [ ]:
# Recordings per species (top and bottom 15)
recs_per_species = train_meta["primary_label"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

recs_per_species.head(20).sort_values().plot.barh(
    ax=axes[0], color="#3498db"
)
axes[0].set_title("Top 20 Species by Recording Count")
axes[0].set_xlabel("Recordings")

recs_per_species.tail(20).sort_values().plot.barh(
    ax=axes[1], color="#e74c3c"
)
axes[1].set_title("Bottom 20 Species by Recording Count")
axes[1].set_xlabel("Recordings")

plt.tight_layout()
plt.show()

print(f"Total species with recordings: {len(recs_per_species)}")
print(f"Median recordings per species: {recs_per_species.median():.0f}")
print(f"Min: {recs_per_species.min()}, Max: {recs_per_species.max()}")


## Data Source and Quality

Recordings come from two collections with different quality characteristics. Xeno-canto provides expert ratings (1-5), while iNaturalist entries have no rating. This quality signal can inform sample weighting during training, though we do not use it in our current probe pipeline.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Collection distribution
coll_counts = train_meta["collection"].value_counts()
coll_counts.plot.pie(
    ax=axes[0], autopct="%1.1f%%",
    colors=["#3498db", "#2ecc71"],
    startangle=90,
)
axes[0].set_title("Recording Source")
axes[0].set_ylabel("")

# Rating distribution (XC only)
xc_ratings = train_meta[train_meta["collection"] == "XC"]["rating"]
xc_ratings[xc_ratings > 0].hist(
    ax=axes[1], bins=10, color="#3498db", edgecolor="white",
)
axes[1].set_title("Xeno-canto Recording Quality Ratings")
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


## Geographic Coverage

Training recordings have latitude/longitude metadata. Plotting these coordinates reveals the geographic spread of the dataset and highlights potential spatial biases.


In [ ]:
geo = train_meta.dropna(subset=["latitude", "longitude"])

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    geo["longitude"], geo["latitude"],
    c="#3498db", alpha=0.15, s=5, edgecolors="none",
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Training Recording Locations ({len(geo):,} recordings)")
plt.tight_layout()
plt.show()


---

# 4. Label Preprocessing and Truth Matrix <a name="labels"></a>

---

Soundscape labels contain semicolon-separated species codes per 5-second window. Some windows have multiple species vocalizing simultaneously. We aggregate labels per window, parse the filename metadata (site, date, hour), identify fully-labeled files, and build a multi-hot truth matrix.


In [ ]:
FNAME_RE = re.compile(
    r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg"
)


def parse_labels(x):
    """Split semicolon-separated label string into a list."""
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]


def union_labels(series):
    """Merge all species labels within a group into a sorted unique set."""
    return sorted(set(lbl for x in series for lbl in parse_labels(x)))


def parse_soundscape_filename(name):
    """Extract site, hour, date from the standardized filename format."""
    m = FNAME_RE.match(name)
    if not m:
        return {"site": None, "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}


# Aggregate labels per 5-second window and attach metadata
sc_clean = (
    soundscape_lbls
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"] = (
    sc_clean["filename"].str.replace(".ogg", "", regex=False)
    + "_" + sc_clean["end_sec"].astype(str)
)
meta_cols = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta_cols], axis=1)

# Identify fully-labeled files (all 12 windows annotated)
wpf = sc_clean.groupby("filename").size()
full_files = sorted(wpf[wpf == CFG.N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

# Build multi-hot label matrix for all soundscape rows
Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean["label_list"]):
    for lbl in labels:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

# Ordered truth for fully-labeled files
full_truth = (
    sc_clean[sc_clean["file_fully_labeled"]]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)
Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]

print(f"Fully-labeled files : {len(full_files)}")
print(f"Trusted windows     : {len(full_truth)}")
print(f"Active classes      : {int((Y_FULL_TRUTH.sum(axis=0) > 0).sum())}")


## Soundscape Species Activity

Of the 234 target species, only a subset appears in the labeled soundscapes. This matters because species absent from soundscape labels cannot have site/hour priors estimated, and those with fewer than `MIN_POS` (8) windows cannot support a per-class probe.


In [ ]:
# Species occurrence in labeled soundscapes
species_activity = Y_FULL_TRUTH.sum(axis=0)
active_mask = species_activity > 0
active_species = [PRIMARY_LABELS[i] for i in np.where(active_mask)[0]]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of positive window counts
active_counts = species_activity[active_mask]
axes[0].hist(active_counts, bins=30, color="#2ecc71", edgecolor="white")
axes[0].set_title(f"Distribution of Positive Windows per Species (n={len(active_counts)})")
axes[0].set_xlabel("Positive Windows")
axes[0].set_ylabel("Number of Species")
axes[0].axvline(x=CFG.MIN_POS, color="red", linestyle="--", label=f"MIN_POS={CFG.MIN_POS}")
axes[0].legend()

# Pie chart: active vs inactive
axes[1].pie(
    [active_mask.sum(), (~active_mask).sum()],
    labels=[f"Active ({active_mask.sum()})", f"Inactive ({(~active_mask).sum()})"],
    colors=["#2ecc71", "#e74c3c"],
    autopct="%1.0f%%",
    startangle=90,
)
axes[1].set_title("Species Presence in Labeled Soundscapes")

plt.tight_layout()
plt.show()


## Site and Temporal Patterns

Species prevalence varies by recording site and time-of-day. This is the spatial-temporal signal captured by our Bayesian prior tables. Sites in different habitats host different species communities, and many species are most vocal at dawn or dusk.


In [ ]:
# Site x hour occurrence matrix
site_hour = sc_clean.groupby(["site", "hour_utc"]).size().reset_index(name="windows")
pivot = site_hour.pivot(index="site", columns="hour_utc", values="windows").fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.5, ax=ax,
)
ax.set_title("Labeled Windows by Site and Hour (UTC)")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Recording Site")
plt.tight_layout()
plt.show()


---

# 5. Perch v2 Model and Species Mapping <a name="perch"></a>

---

Google Perch v2 was pretrained on 14,795 species identified by scientific name. Each of the 234 competition species is mapped to its Perch class index by joining on `scientific_name`. Species with no direct match are left unmapped or, for unmapped amphibians, assigned a genus-level proxy from any Perch class sharing the same genus. The max logit across genus matches serves as the proxy score.


In [ ]:
print("Loading Perch v2 model...")
birdclassifier = tf.saved_model.load(str(CFG.MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]
print("Perch v2 loaded.")

# Build species-to-Perch-index mapping
bc_labels = (
    pd.read_csv(CFG.MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)
NO_LABEL_INDEX = len(bc_labels)

taxonomy_ = taxonomy.copy()
taxonomy_["scientific_name"] = taxonomy_["scientific_name"].astype(str)
mapping = taxonomy_.merge(
    bc_labels[["scientific_name", "bc_index"]],
    on="scientific_name", how="left",
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK       = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS        = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS      = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped   : {MAPPED_MASK.sum()} / {N_CLASSES}")
print(f"Unmapped : {(~MAPPED_MASK).sum()}")


In [ ]:
# Split species into texture classes (frogs, insects) and event classes (birds, etc)
CLASS_NAME_MAP = taxonomy_.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
    dtype=np.int32,
)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA],
    dtype=np.int32,
)

idx_mapped_active_texture  = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event    = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event   = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES],
    dtype=np.int32,
)

# Build genus-level proxies for unmapped amphibians
# If a species is not in Perch's vocabulary but a congener is,
# we use the max logit across all congener indices as a proxy score
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    genus = str(row["scientific_name"]).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].str.match(rf"^{re.escape(genus)}\s", na=False)
    ]
    if len(hits) > 0:
        proxy_map[str(row["primary_label"])] = hits["bc_index"].astype(int).tolist()

SELECTED_PROXY_TARGETS = sorted(
    [t for t in proxy_map if CLASS_NAME_MAP.get(t) == "Amphibia"]
)
selected_proxy_pos = np.array(
    [label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32
)
selected_proxy_pos_to_bc = {
    label_to_idx[t]: np.array(proxy_map[t], dtype=np.int32)
    for t in SELECTED_PROXY_TARGETS
}

idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f"Active texture classes : {len(idx_active_texture)}")
print(f"Active event classes   : {len(idx_active_event)}")
print(f"Frog proxy targets     : {SELECTED_PROXY_TARGETS}")


---

# 6. Perch Inference Engine <a name="engine"></a>

---

Each 60-second soundscape is split into 12 non-overlapping 5-second windows and passed through Perch v2 in batches. The model returns both raw logits over its 14k vocabulary and a 1536-dimensional embedding per window. We extract the logits for our 203 mapped species and compute genus-proxy scores for the 3 unmapped amphibians.


In [ ]:
def read_soundscape_60s(path):
    """Load a 60-second soundscape file, padding or truncating to exact length."""
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < CFG.FILE_SAMPLES:
        y = np.pad(y, (0, CFG.FILE_SAMPLES - len(y)))
    return y[:CFG.FILE_SAMPLES]


def infer_perch_batch(paths, verbose=True):
    """
    Run Perch v2 inference on a list of soundscape file paths.
    Returns metadata DataFrame, raw score matrix, and embedding matrix.
    """
    paths   = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows  = n_files * CFG.N_WINDOWS

    row_ids    = np.empty(n_rows, dtype=object)
    filenames  = np.empty(n_rows, dtype=object)
    sites      = np.empty(n_rows, dtype=object)
    hours      = np.empty(n_rows, dtype=np.int16)
    scores     = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536),      dtype=np.float32)

    write_row = 0
    itr = tqdm(
        range(0, n_files, CFG.BATCH_FILES),
        desc="Perch", disable=not verbose,
    )

    for start in itr:
        batch  = paths[start : start + CFG.BATCH_FILES]
        bn     = len(batch)
        x      = np.empty((bn * CFG.N_WINDOWS, CFG.WINDOW_SAMPLES), dtype=np.float32)
        bstart = write_row

        for bi, path in enumerate(batch):
            audio = read_soundscape_60s(path)
            x[bi * CFG.N_WINDOWS : (bi + 1) * CFG.N_WINDOWS] = audio.reshape(
                CFG.N_WINDOWS, CFG.WINDOW_SAMPLES
            )
            meta = parse_soundscape_filename(path.name)
            row_ids[write_row : write_row + CFG.N_WINDOWS] = [
                f"{path.stem}_{t}" for t in range(5, 65, 5)
            ]
            filenames[write_row : write_row + CFG.N_WINDOWS] = path.name
            sites[write_row : write_row + CFG.N_WINDOWS]     = meta["site"]
            hours[write_row : write_row + CFG.N_WINDOWS]     = meta["hour_utc"]
            write_row += CFG.N_WINDOWS

        out    = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = out["label"].numpy().astype(np.float32)
        emb    = out["embedding"].numpy().astype(np.float32)

        # Extract mapped species logits
        scores[bstart:write_row, MAPPED_POS] = logits[: write_row - bstart, MAPPED_BC_INDICES]
        embeddings[bstart:write_row] = emb

        # Genus proxies: max logit across congener indices
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            scores[bstart:write_row, pos] = logits[: write_row - bstart, bc_idx_arr].max(axis=1)

        del x, out, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        "row_id": row_ids, "filename": filenames,
        "site": sites, "hour_utc": hours,
    })
    return meta_df, scores, embeddings


## Load Cache or Run Inference

Running Perch on 59 files takes roughly 12 minutes. To stay within the 90-minute submission budget, pre-computed outputs are attached as a Kaggle dataset. If the cache is missing (first run or local dev), inference runs and the outputs are saved for reuse.


In [ ]:
if CFG.CACHE_DIR is not None:
    print(f"Loading Perch cache from: {CFG.CACHE_DIR}")
    meta_full       = pd.read_parquet(CFG.CACHE_DIR / "full_perch_meta.parquet")
    arr             = np.load(CFG.CACHE_DIR / "full_perch_arrays.npz")
    scores_full_raw = arr["scores_full_raw"].astype(np.float32)
    emb_full        = arr["emb_full"].astype(np.float32)
else:
    print("No cache found. Running Perch on fully-labeled soundscapes...")
    full_paths = [CFG.BASE / "train_soundscapes" / fn for fn in full_files]
    meta_full, scores_full_raw, emb_full = infer_perch_batch(full_paths)
    meta_full.to_parquet(CFG.WORK_CACHE / "full_perch_meta.parquet", index=False)
    np.savez_compressed(
        CFG.WORK_CACHE / "full_perch_arrays.npz",
        scores_full_raw=scores_full_raw,
        emb_full=emb_full,
    )
    print(f"Cache saved to {CFG.WORK_CACHE}")

# Align ground truth to cache row order
full_truth_aligned = (
    full_truth.set_index("row_id")
    .loc[meta_full["row_id"]]
    .reset_index(drop=False)
)
Y_FULL = Y_SC[full_truth_aligned["index"].to_numpy()]

print(f"scores_full_raw : {scores_full_raw.shape}  {scores_full_raw.dtype}")
print(f"emb_full        : {emb_full.shape}  {emb_full.dtype}")
print(f"Y_FULL          : {Y_FULL.shape}")


---

# 7. Bayesian Prior Fusion <a name="priors"></a>

---

Raw Perch logits reflect global species probabilities from pretraining but ignore the deployment context. Bayesian prior tables are built from the soundscape metadata, covering site prevalence, hour-of-day prevalence, and their joint prevalence. Each estimate is shrunk toward the global mean using a sample-size-dependent mixing weight, which protects against small-sample noise. The resulting logit offsets are added to the raw scores with class-type-specific lambdas.


In [ ]:
def fit_prior_tables(prior_df, Y_prior):
    """Build site, hour, and site-hour prevalence tables."""
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)

    # Site prevalence
    site_keys = sorted(prior_df["site"].dropna().astype(str).unique())
    site_to_i, site_n, site_p = {}, [], []
    for s in site_keys:
        mask = prior_df["site"].astype(str).values == s
        site_to_i[s] = len(site_n)
        site_n.append(mask.sum())
        site_p.append(Y_prior[mask].mean(axis=0))
    site_n = np.array(site_n, dtype=np.float32)
    site_p = np.stack(site_p).astype(np.float32) if site_p else np.zeros((0, Y_prior.shape[1]), np.float32)

    # Hour prevalence
    hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique())
    hour_to_i, hour_n, hour_p = {}, [], []
    for h in hour_keys:
        mask = prior_df["hour_utc"].astype(int).values == h
        hour_to_i[h] = len(hour_n)
        hour_n.append(mask.sum())
        hour_p.append(Y_prior[mask].mean(axis=0))
    hour_n = np.array(hour_n, dtype=np.float32)
    hour_p = np.stack(hour_p).astype(np.float32) if hour_p else np.zeros((0, Y_prior.shape[1]), np.float32)

    # Site-hour joint prevalence
    sh_to_i, sh_n_list, sh_p_list = {}, [], []
    for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))
    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, Y_prior.shape[1]), np.float32)

    return dict(
        global_p=global_p,
        site_to_i=site_to_i, site_n=site_n, site_p=site_p,
        hour_to_i=hour_to_i, hour_n=hour_n, hour_p=hour_p,
        sh_to_i=sh_to_i,     sh_n=sh_n,     sh_p=sh_p,
    )


def prior_logits(sites, hours, tables, eps=1e-4):
    """Convert prevalence estimates to log-odds with hierarchical shrinkage."""
    n = len(sites)
    p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)

    si  = np.fromiter((tables["site_to_i"].get(str(s), -1) for s in sites), np.int32, n)
    hi  = np.fromiter(
        (tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
        np.int32, n,
    )
    shi = np.fromiter(
        (tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1
         for s, h in zip(sites, hours)),
        np.int32, n,
    )

    # Each level refines the estimate with sample-size-based mixing
    valid = hi >= 0
    if valid.any():
        nh = tables["hour_n"][hi[valid]][:, None]
        p[valid] = nh / (nh + 8) * tables["hour_p"][hi[valid]] + (1 - nh / (nh + 8)) * p[valid]

    valid = si >= 0
    if valid.any():
        ns = tables["site_n"][si[valid]][:, None]
        p[valid] = ns / (ns + 8) * tables["site_p"][si[valid]] + (1 - ns / (ns + 8)) * p[valid]

    valid = shi >= 0
    if valid.any():
        nsh = tables["sh_n"][shi[valid]][:, None]
        p[valid] = nsh / (nsh + 4) * tables["sh_p"][shi[valid]] + (1 - nsh / (nsh + 4)) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)


In [ ]:
def smooth_cols(scores, cols, alpha=0.35):
    """Temporal smoothing: blend each window with neighbors within the same file.
    Used for texture species (frogs, insects) whose presence spans multiple windows."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s    = scores.copy()
    view = s.reshape(-1, CFG.N_WINDOWS, s.shape[1])
    x    = view[:, :, cols]
    prev = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    nxt  = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev + nxt)
    return s


def smooth_events(scores, cols, alpha=0.15):
    """Soft max-pool context for event species (birds).
    Uses local_max instead of average neighbor, which preserves transient calls."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s    = scores.copy()
    view = s.reshape(-1, CFG.N_WINDOWS, s.shape[1])
    x    = view[:, :, cols]
    prev = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    nxt  = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev, nxt))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s


def fuse_scores(base, sites, hours, tables):
    """Combine raw Perch logits with Bayesian prior offsets."""
    scores = base.copy()
    prior  = prior_logits(sites, hours, tables)

    # Mapped active species: add prior with class-type-specific weight
    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += CFG.LAMBDA_EVENT * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += CFG.LAMBDA_TEXTURE * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += CFG.LAMBDA_PROXY_TEXTURE * prior[:, idx_selected_proxy_active_texture]

    # Unmapped active species: prior is the only signal
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = CFG.LAMBDA_EVENT * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = CFG.LAMBDA_TEXTURE * prior[:, idx_selected_prioronly_active_texture]

    # Unmapped inactive species: suppress to avoid false positives
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    # Temporal smoothing with different strategies per class type
    scores = smooth_cols(scores, idx_active_texture, alpha=CFG.SMOOTH_TEXTURE)
    scores = smooth_events(scores, idx_active_event, alpha=CFG.SMOOTH_EVENT)
    return scores.astype(np.float32), prior


---

# 8. Out-of-Fold Meta-Features <a name="oof"></a>

---

The embedding probes take fused scores as input features, so those scores must be produced out-of-fold to avoid data leakage. The prior table for each validation fold is fitted on training folds only, excluding the entire validation-file set. `GroupKFold(5)` with groups defined by filename prevents any file from contributing to both training and validation.


In [ ]:
def build_oof_base_prior(scores_raw, meta_df, sc_clean, Y_SC, n_splits=5, verbose=True):
    """Build honest out-of-fold fused scores and prior logits."""
    groups = meta_df["filename"].to_numpy()
    gkf = GroupKFold(n_splits=n_splits)

    oof_base  = np.zeros_like(scores_raw, dtype=np.float32)
    oof_prior = np.zeros_like(scores_raw, dtype=np.float32)
    fold_id   = np.full(len(meta_df), -1, dtype=np.int16)

    splits = list(gkf.split(scores_raw, groups=groups))
    itr = tqdm(splits, desc="OOF folds", disable=not verbose)

    for fold, (tr_idx, va_idx) in enumerate(itr, 1):
        tr_idx = np.sort(tr_idx)
        va_idx = np.sort(va_idx)

        val_files = set(meta_df.iloc[va_idx]["filename"].tolist())

        # Fit prior tables excluding validation files
        prior_mask = ~sc_clean["filename"].isin(val_files).values
        tables = fit_prior_tables(
            sc_clean.loc[prior_mask].reset_index(drop=True),
            Y_SC[prior_mask],
        )

        va_base, va_prior = fuse_scores(
            scores_raw[va_idx],
            sites=meta_df.iloc[va_idx]["site"].to_numpy(),
            hours=meta_df.iloc[va_idx]["hour_utc"].to_numpy(),
            tables=tables,
        )

        oof_base[va_idx]  = va_base
        oof_prior[va_idx] = va_prior
        fold_id[va_idx]   = fold

    assert (fold_id >= 0).all()
    return oof_base, oof_prior, fold_id


def macro_auc_skip_empty(y_true, y_score):
    """Macro-averaged ROC-AUC, skipping classes with no positive labels."""
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")


# Build or load OOF meta-features
OOF_CACHE = CFG.WORK_CACHE / "oof_meta.npz"

if OOF_CACHE.exists():
    print(f"Loading cached OOF from: {OOF_CACHE}")
    arr = np.load(OOF_CACHE)
    oof_base  = arr["oof_base"].astype(np.float32)
    oof_prior = arr["oof_prior"].astype(np.float32)
    oof_fold_id = arr["fold_id"].astype(np.int16)
else:
    print("Building OOF meta-features...")
    oof_base, oof_prior, oof_fold_id = build_oof_base_prior(
        scores_full_raw, meta_full, sc_clean, Y_SC,
        n_splits=5, verbose=(CFG.MODE == "train"),
    )
    np.savez_compressed(OOF_CACHE, oof_base=oof_base, oof_prior=oof_prior, fold_id=oof_fold_id)
    print(f"Saved OOF to: {OOF_CACHE}")

baseline_oof_auc = macro_auc_skip_empty(Y_FULL, oof_base)
print(f"OOF baseline AUC: {baseline_oof_auc:.6f}")


---

# 9. Embedding Probes <a name="probes"></a>

---

Per-class MLP probes are trained on PCA-compressed Perch embeddings augmented with temporal context features. The probes learn species-specific decision boundaries in the embedding space that complement the Perch logits. Temporal features (previous/next window scores, file-level mean/max/std) provide sequence-level context that a per-window classifier otherwise lacks.


In [ ]:
def seq_features_1d(v):
    """Extract temporal context features from a per-window score vector.
    Provides previous/next neighbor values, file-level mean, max, and std."""
    assert len(v) % CFG.N_WINDOWS == 0
    x = v.reshape(-1, CFG.N_WINDOWS)

    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), CFG.N_WINDOWS)
    max_v  = np.repeat(x.max(axis=1),  CFG.N_WINDOWS)
    std_v  = np.repeat(x.std(axis=1),  CFG.N_WINDOWS)

    return prev_v, next_v, mean_v, max_v, std_v


def build_class_features(emb_proj, raw_col, prior_col, base_col):
    """Assemble the full feature vector for a single species classifier.
    Combines PCA-reduced embedding with score signals and temporal context."""
    prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base_col)

    # Onset/offset detection: is this window higher than its neighbors?
    diff_mean = base_col - mean_base
    diff_prev = base_col - prev_base
    diff_next = base_col - next_base

    feats = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
        std_base[:, None],
        diff_mean[:, None],
        diff_prev[:, None],
        diff_next[:, None],
        # Interaction terms capture joint signal strength
        (raw_col * prior_col)[:, None],
        (raw_col * base_col)[:, None],
        (prior_col * base_col)[:, None],
    ], axis=1)

    return feats.astype(np.float32, copy=False)


In [ ]:
# Fit embedding PCA on all trusted windows
emb_scaler = StandardScaler()
emb_full_scaled = emb_scaler.fit_transform(emb_full)

n_comp = min(CFG.PCA_DIM, emb_full_scaled.shape[0] - 1, emb_full_scaled.shape[1])
emb_pca = PCA(n_components=n_comp)
Z_FULL = emb_pca.fit_transform(emb_full_scaled).astype(np.float32)

print(f"Embedding PCA: {emb_full.shape[1]}d -> {Z_FULL.shape[1]}d")
print(f"Explained variance: {emb_pca.explained_variance_ratio_.sum():.4f}")

# Train per-class MLP probes on OOF meta-features
full_pos_counts = Y_FULL.sum(axis=0)
PROBE_CLASS_IDX = np.where(full_pos_counts >= CFG.MIN_POS)[0].astype(np.int32)

probe_models = {}

for cls_idx in tqdm(PROBE_CLASS_IDX, desc="Training probes", disable=(CFG.MODE != "train")):
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y):
        continue

    X_cls = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=oof_prior[:, cls_idx],
        base_col=oof_base[:, cls_idx],
    )

    # Oversample positives to handle class imbalance
    n_pos = int(y.sum())
    n_neg = len(y) - n_pos
    if n_pos > 0 and n_neg > n_pos:
        repeat = max(1, n_neg // n_pos)
        pos_idx = np.where(y == 1)[0]
        X_bal = np.vstack([X_cls, np.tile(X_cls[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(len(pos_idx) * repeat, dtype=y.dtype)])
    else:
        X_bal, y_bal = X_cls, y

    clf = MLPClassifier(**CFG.MLP_PARAMS)
    clf.fit(X_bal, y_bal)
    probe_models[cls_idx] = clf

print(f"Trained probes: {len(probe_models)} species")


---

# 10. Test Inference and Submission <a name="submit"></a>

---

The hidden test set is loaded, Perch inference is run, prior fusion is applied, and per-class probe predictions are blended with the fused base scores. The final probabilities are written to `submission.csv`.


In [ ]:
# Detect test soundscapes
test_paths = sorted((CFG.BASE / "test_soundscapes").glob("*.ogg"))

if len(test_paths) == 0:
    print(f"Hidden test not mounted. Dry-run on first {CFG.DRYRUN_FILES} train soundscapes.")
    test_paths = sorted((CFG.BASE / "train_soundscapes").glob("*.ogg"))[:CFG.DRYRUN_FILES]
else:
    print(f"Hidden test files: {len(test_paths)}")

# Run Perch inference on test soundscapes
meta_test, scores_test_raw, emb_test = infer_perch_batch(
    test_paths, verbose=(CFG.MODE == "train"),
)

print(f"meta_test       : {meta_test.shape}")
print(f"scores_test_raw : {scores_test_raw.shape}")
print(f"emb_test        : {emb_test.shape}")


In [ ]:
# Build final prior tables on all labeled soundscapes
final_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)

# Fuse test scores with priors
scores_test_fused, prior_test = fuse_scores(
    scores_test_raw,
    sites=meta_test["site"].to_numpy(),
    hours=meta_test["hour_utc"].to_numpy(),
    tables=final_tables,
)

# Apply probe predictions
emb_test_scaled = emb_scaler.transform(emb_test)
Z_test = emb_pca.transform(emb_test_scaled).astype(np.float32)

scores_final = scores_test_fused.copy()

for cls_idx, clf in probe_models.items():
    X_test_cls = build_class_features(
        Z_test,
        raw_col=scores_test_raw[:, cls_idx],
        prior_col=prior_test[:, cls_idx],
        base_col=scores_test_fused[:, cls_idx],
    )

    # Convert MLP probability to logit space for blending
    pred_prob = clf.predict_proba(X_test_cls)[:, 1].astype(np.float32)
    pred_logit = np.log(pred_prob + 1e-7) - np.log(1 - pred_prob + 1e-7)

    scores_final[:, cls_idx] = (
        (1.0 - CFG.PROBE_ALPHA) * scores_test_fused[:, cls_idx]
        + CFG.PROBE_ALPHA * pred_logit
    )

print(f"Probes applied to {len(probe_models)} species columns.")


In [ ]:
# Convert logits to probabilities and build submission
probs = 1.0 / (1.0 + np.exp(np.clip(-scores_final, -50, 50)))

submission = pd.DataFrame(probs, columns=PRIMARY_LABELS)
submission.insert(0, "row_id", meta_test["row_id"].values)

# Align with expected submission format
expected = sample_sub.columns.tolist()
submission = submission[expected]

submission.to_csv("submission.csv", index=False)

print(f"Submission shape: {submission.shape}")
print(f"Saved to: submission.csv")
submission.head(3)


---

## Summary

This notebook implements a multi-stage pipeline for acoustic species identification in the Pantanal:

1. **Perch v2** extracts 1536-d embeddings and 234-class logits from 5-second audio windows.
2. **Bayesian prior fusion** incorporates site and time-of-day prevalence with class-type-specific weights.
3. **MLP probes** on PCA-compressed embeddings learn per-species decision boundaries.
4. **Temporal smoothing** (neighbor-average for texture species, local-max for event species) reduces false negatives.

The final submission blends fused Perch logits with probe predictions using a tuned mixing weight.

---

**Citation**: Stefan Kahl, Tom Denton, Larissa Sugai, Liliana Piatti, Ryan Holbrook, Holger Klinck, and Ashley Oldacre. BirdCLEF+ 2026. https://kaggle.com/competitions/birdclef-2026, 2026. Kaggle.
